In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_order_items = spark.read.table('global_partner.silver.order_items')
df_order_item_options = spark.read.table('global_partner.silver.order_item_options')
df_date_dim = spark.read.table('global_partner.bronze.date_dim')

# Churn Indicators

In [0]:
df_order_items = df_order_items.withColumn('CREATION_TIME_UTC',F.to_timestamp('CREATION_TIME_UTC'))
df_churn = df_order_items.select('CREATION_TIME_UTC','USER_ID','ITEM_PRICE','ORDER_ID').distinct()
df_churn.display()

In [0]:
snapshot_date = (df_churn.agg(F.max('CREATION_TIME_UTC').alias('snapshot_date')).first()['snapshot_date'])
#agg_df = df_churn.groupBy('USER_ID').agg(F.max('CREATION_TIME_UTC').alias('max_creation_time'))
window = Window.partitionBy('user_id')
df_churn = df_churn.withColumn('max_creation_time',F.max('CREATION_TIME_UTC').over(window))
df_churn = df_churn.withColumn('Days_since_last_order',F.datediff(F.lit(snapshot_date),F.col('max_creation_time')))
df_churn.display()

In [0]:
window1 = Window.partitionBy('user_id').orderBy('CREATION_TIME_UTC')
df_churn = df_churn.withColumn('prev_order_dt',F.lag('CREATION_TIME_UTC').over(window1))
df_churn = df_churn.withColumn('time_bw_order',F.datediff(F.col('CREATION_TIME_UTC'),F.col('prev_order_dt')))
df_churn = df_churn.withColumn('average_gap',F.avg('time_bw_order').over(window))
df_churn = df_churn.withColumn('prev_spend',F.lag('ITEM_PRICE').over(window1))

df_churn.display()

# Sales Trend Monitoring

In [0]:
df_order_items = spark.read.table('global_partner.silver.order_items')
df_order_items = df_order_items.select('CREATION_TIME_UTC','ORDER_ID','ITEM_CATEGORY','ITEM_PRICE').distinct()
df_order_items = df_order_items.withColumn('CREATION_TIME_UTC',F.to_timestamp(F.col('CREATION_TIME_UTC')))
df_order_items = df_order_items.withColumn('CREATION_DATE_UTC',F.to_date(F.col('CREATION_TIME_UTC')))
df_order_items = df_order_items.withColumn('CREATION_YEAR',F.year(F.col('CREATION_DATE_UTC')))
df_order_items = df_order_items.withColumn('CREATION_MONTH',F.month(F.col('CREATION_DATE_UTC')))
df_order_items = df_order_items.withColumn('CREATION_WEEK',F.weekofyear(F.col('CREATION_DATE_UTC')))
df_order_items.display()

In [0]:
daily_agg_df = df_order_items.groupBy('CREATION_DATE_UTC').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy('CREATION_DATE_UTC')
daily_agg_df.display()

In [0]:
weekly_agg_df = df_order_items.groupBy('CREATION_YEAR','CREATION_MONTH','CREATION_WEEK').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy('CREATION_YEAR','CREATION_MONTH','CREATION_WEEK')
weekly_agg_df.display()

In [0]:
monthly_agg_df = df_order_items.groupBy('CREATION_YEAR','CREATION_MONTH').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy('CREATION_YEAR','CREATION_MONTH')
monthly_agg_df.display()

In [0]:
agg_df = df_order_items.groupBy('ITEM_CATEGORY').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy(F.col('total_revenue').desc())
agg_df.display()

In [0]:
df_order_items = spark.read.table('global_partner.silver.order_items')
df_date_dim = spark.read.table('global_partner.bronze.date_dim')
df_order_items = df_order_items.withColumn('CREATION_TIME_UTC',F.to_timestamp(F.col('CREATION_TIME_UTC')))
df_order_items = df_order_items.withColumn('CREATION_DATE_UTC',F.to_date(F.col('CREATION_TIME_UTC')))
df_order_items.printSchema()
df_date_dim.printSchema()
df_order_items.display()


In [0]:
df_date_dim.display()

In [0]:
df_order_items = df_order_items.select('CREATION_TIME_UTC','CREATION_DATE_UTC','ORDER_ID','ITEM_CATEGORY','ITEM_PRICE').distinct()

In [0]:
df_order_items.count()

In [0]:
df = df_order_items.join(df_date_dim,
                          df_order_items['CREATION_DATE_UTC']==df_date_dim['date_key'],
                          how='left')

In [0]:
df.display()
df.count()

In [0]:
daily_agg_df = df.groupBy('CREATION_DATE_UTC').agg(F.round(F.sum('item_price'),3).alias('total_revenue')).orderBy('CREATION_DATE_UTC')
daily_agg_df.display()